# Side-Step — Train `[slim]`

**Preprocessing + training only.** For the full pipeline (captions, audio analysis, tags, export, history), use [Side_Step_Colab.ipynb](https://colab.research.google.com/github/koda-dernet/Side-Step-Colab/blob/main/Side_Step_Colab.ipynb).

> Run cells top-to-bottom. Every `#@param` field has a hover tooltip.

| Section | Description |
|---------|-------------|
| S0 GPU Check | Verify GPU runtime |
| S1 Install & Mount | Clone Side-Step, mount Drive |
| S2 Model Downloads | Download ACE-Step checkpoints |
| S3 Preprocessing | Audio to .pt tensors |
| S4 PP++ Analysis | Adaptive rank assignment *(optional)* |
| S5 Training | Train adapter |


---
## S0 — GPU Check
> If you see red, go to **Runtime > Change runtime type > GPU**.


In [ ]:
import torch
from IPython.display import HTML, display

_SS = {
    "bg":"#16161E","surface":"#1E1F28","panel":"#2D2E36","border":"#4A5F8F",
    "primary":"#5ccfe6","success":"#7ec97a","warning":"#ffcc66","error":"#f07178",
    "text":"#A1A8CF","bold":"#D5D8EC","muted":"#8995BC",
}

def _card(title, body, accent="primary"):
    c = _SS[accent]
    return HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {c};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{c};font-weight:bold;margin-bottom:6px;">{title}</div>'
        f'<div style="color:{_SS["text"]};">{body}</div></div>'
    )

def _table(rows):
    r = "".join(
        f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">{k}</td>'
        f'<td style="padding:3px 0;color:{_SS["bold"]};font-weight:600;">{v}</td></tr>'
        for k,v in rows
    )
    return f'<table style="border-collapse:collapse;">{r}</table>'

if not torch.cuda.is_available():
    display(_card("[FAIL] No GPU Detected",
        "Go to <b>Runtime > Change runtime type > GPU</b> and re-run this cell.", "error"))
    raise RuntimeError("No GPU runtime.")
else:
    _n = torch.cuda.get_device_name(0)
    _v = torch.cuda.get_device_properties(0).total_mem / 1024**3
    _c = torch.version.cuda or "N/A"
    display(_card("[OK] GPU Ready", _table([
        ("GPU", _n), ("VRAM", f"{_v:.1f} GB"), ("CUDA", _c), ("PyTorch", torch.__version__)
    ]), "success"))


---
## S1 — Install & Mount Drive
> Everything is saved to `My Drive/SideStep/` and survives runtime disconnects.


In [ ]:
#@title S1 -- Install Side-Step & Mount Google Drive { display-mode: "form" }

import os, subprocess, sys
from pathlib import Path
from IPython.display import HTML, display

REPO_URL = "https://github.com/koda-dernet/Side-Step.git"
BRANCH = "main"  #@param {type:"string"} {tooltip: "Branch to clone. Use 'main' for stable releases."}
INSTALL_DIR = "/content/Side-Step"

if not Path(INSTALL_DIR).exists():
    print("[INFO] Cloning Side-Step (%s)..." % BRANCH)
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, INSTALL_DIR],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    print("[OK] Cloned.")
else:
    print("[OK] Side-Step already present at %s" % INSTALL_DIR)

print("[INFO] Installing dependencies (takes ~2 min on first run)...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{INSTALL_DIR}/requirements.txt"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
)
print("[OK] Dependencies installed.")

if INSTALL_DIR not in sys.path:
    sys.path.insert(0, INSTALL_DIR)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

SIDESTEP_ROOT = "/content/drive/MyDrive/SideStep"  #@param {type:"string"} {tooltip: "Root directory on Google Drive for all Side-Step files."}

for name in ("checkpoints", "tensors", "adapters", "logs"):
    Path(f"{SIDESTEP_ROOT}/{name}").mkdir(parents=True, exist_ok=True)

from sidestep_engine.ui import set_plain_mode
set_plain_mode(True)
import torch
torch.set_float32_matmul_precision("medium")

# Save minimal settings
from sidestep_engine.settings import load_settings, save_settings, _default_settings
settings = load_settings() or _default_settings()
settings["checkpoint_dir"] = os.path.join(SIDESTEP_ROOT, "checkpoints")
settings["trained_adapters_dir"] = os.path.join(SIDESTEP_ROOT, "adapters")
settings["preprocessed_tensors_dir"] = os.path.join(SIDESTEP_ROOT, "tensors")
settings["first_run_complete"] = True
save_settings(settings)

print("[OK] Side-Step is ready.")


---
## S2 — Model Downloads
Download ACE-Step checkpoints to Drive. These persist across sessions.

| Model | HF Repo | Contents |
|-------|---------|----------|
| **ACE-Step 1.5 (shared)** | `ACE-Step/Ace-Step1.5` | Turbo variant, Qwen text embeddings, VAE |
| **ACE-Step base** | `ACE-Step/acestep-v15-base` | Base variant *(recommended for training)* |
| **ACE-Step SFT** | `ACE-Step/acestep-v15-sft` | SFT variant |


In [ ]:
#@title S2 -- Download Models { display-mode: "form" }
#@markdown Already-downloaded models are skipped.

download_shared = True   #@param {type:"boolean"} {tooltip: "ACE-Step shared weights (turbo, VAE, Qwen embeddings). Always required."}
download_base   = True   #@param {type:"boolean"} {tooltip: "ACE-Step base variant. Recommended for training."}
download_sft    = False  #@param {type:"boolean"} {tooltip: "ACE-Step SFT variant."}

#@markdown ---
#@markdown ### Model Variant
model_variant = "base"  #@param ["base", "sft", "turbo"] {tooltip: "'base' recommended -- produces the best LoRAs."}

import os
from pathlib import Path
from IPython.display import HTML, display

CHECKPOINT_DIR = os.path.join(SIDESTEP_ROOT, "checkpoints")
_dl = []
if download_shared: _dl.append(("ACE-Step/Ace-Step1.5", "Ace-Step1.5"))
if download_base:   _dl.append(("ACE-Step/acestep-v15-base", "acestep-v15-base"))
if download_sft:    _dl.append(("ACE-Step/acestep-v15-sft", "acestep-v15-sft"))

_SS = {"panel":"#2D2E36","border":"#4A5F8F","success":"#7ec97a","error":"#f07178","text":"#A1A8CF"}

if not _dl:
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["error"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["error"]};font-weight:bold;">[WARN] No models selected.</div></div>'
    ))
else:
    from huggingface_hub import snapshot_download
    for repo_id, folder_name in _dl:
        local_dir = os.path.join(CHECKPOINT_DIR, folder_name)
        if Path(local_dir).is_dir() and any(Path(local_dir).iterdir()):
            print(f"[OK] {repo_id} -- already present")
            continue
        print(f"[INFO] Downloading {repo_id}...")
        snapshot_download(repo_id=repo_id, local_dir=local_dir, local_dir_use_symlinks=False)
        print(f"[OK] {repo_id} -- done")

    found = [d.name for d in Path(CHECKPOINT_DIR).iterdir() if d.is_dir()]
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["success"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["success"]};font-weight:bold;">[OK] Checkpoints Ready</div>'
        f'<div style="color:{_SS["text"]};">Found: <b>{", ".join(found)}</b></div></div>'
    ))


---
## S3 — Preprocessing
Convert raw audio into `.pt` tensor files (two-pass: VAE encoder, then DIT encoder).

> Models load sequentially to minimize VRAM. Output tensors on Drive can be reused across runs.
>
> **Already have `.pt` tensors?** Skip to S5.


In [ ]:
#@title S3 -- Preprocess Audio to Tensors { display-mode: "form" }
#@markdown Two-pass pipeline: VAE encodes waveforms, then DIT encoder produces training-ready tensors.

#@markdown ### Paths
preprocess_audio_dir    = "/content/drive/MyDrive/SideStep/audio"              #@param {type:"string"} {tooltip: "Directory with raw audio files (.wav, .mp3, .flac, .ogg, .m4a)."}
preprocess_output_dir   = "/content/drive/MyDrive/SideStep/tensors/my_dataset" #@param {type:"string"} {tooltip: "Where to save .pt tensors."}
preprocess_dataset_json = ""  #@param {type:"string"} {tooltip: "Optional dataset.json. Leave blank to read .txt sidecars directly."}

#@markdown ### Options
normalize_mode = "peak"  #@param ["none", "peak", "lufs"] {tooltip: "'peak' = -1.0 dBFS (recommended). 'lufs' = -14 LUFS. 'none' = skip."}
max_duration   = 0       #@param {type:"integer"} {tooltip: "Max audio duration in seconds. 0 = auto-detect."}

import os
from IPython.display import HTML, display
from sidestep_engine.data.preprocess import preprocess_audio_files
from sidestep_engine.models.gpu_utils import clear_device_cache

print("=" * 60)
print("  Preprocessing")
print("=" * 60)
print(f"  Source:     {preprocess_audio_dir}")
if preprocess_dataset_json: print(f"  JSON:       {preprocess_dataset_json}")
print(f"  Output:     {preprocess_output_dir}")
print(f"  Variant:    {model_variant}")
print(f"  Normalize:  {normalize_mode}")
print(f"  Max dur:    {'auto' if max_duration <= 0 else f'{max_duration}s'}")
print("=" * 60)

_preprocessed_tensor_dir = preprocess_output_dir
try:
    result = preprocess_audio_files(
        audio_dir=preprocess_audio_dir, output_dir=preprocess_output_dir,
        checkpoint_dir=CHECKPOINT_DIR, variant=model_variant,
        max_duration=max_duration, dataset_json=preprocess_dataset_json or None,
        device="cuda:0", precision="auto", normalize=normalize_mode,
        target_db=-1.0, target_lufs=-14.0,
    )
    _preprocessed_tensor_dir = result["output_dir"]
    _SS = {"panel":"#2D2E36","border":"#4A5F8F","success":"#7ec97a","error":"#f07178","text":"#A1A8CF","bold":"#D5D8EC"}
    _fail = f'<div style="color:{_SS["error"]};margin-top:4px;">Failed: <b>{result["failed"]}</b></div>' if result.get("failed") else ""
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["success"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["success"]};font-weight:bold;">[OK] Preprocessing Complete</div>'
        f'<div style="color:{_SS["text"]};">Processed: <b>{result["processed"]}/{result["total"]}</b></div>'
        f'{_fail}'
        f'<div style="color:{_SS["text"]};margin-top:4px;">Output: <code>{result["output_dir"]}</code></div></div>'
    ))
except Exception as exc:
    print(f"\n[FAIL] Preprocessing failed: {exc}")
    import traceback; traceback.print_exc()
finally:
    clear_device_cache("cuda:0")


---
## S4 — PP++ / Fisher Analysis `[OPTIONAL]`
Compute per-module importance and assign adaptive LoRA ranks.

> PP++ assigns higher ranks to important layers, lower ranks to unimportant ones. Only compatible with LoRA and DoRA.


In [ ]:
#@title S4 -- PP++ Fisher Analysis { display-mode: "form" }
#@markdown Adaptive rank assignment based on your dataset.

run_pp_analysis   = False      #@param {type:"boolean"} {tooltip: "Run PP++ analysis. Saves fisher_map.json alongside tensors."}
pp_base_rank      = 64         #@param {type:"integer"} {tooltip: "Base rank budget."}
pp_rank_min       = 16         #@param {type:"integer"} {tooltip: "Minimum rank for any module."}
pp_rank_max       = 128        #@param {type:"integer"} {tooltip: "Maximum rank for any module."}
pp_timestep_focus = "balanced" #@param ["balanced", "early", "mid", "late"] {tooltip: "'balanced' = all timesteps equally."}

#@markdown ---
#@markdown Dataset auto-filled from S3.

if run_pp_analysis:
    _ds = _preprocessed_tensor_dir if "_preprocessed_tensor_dir" in dir() else preprocess_output_dir
    from sidestep_engine.analysis.fisher import run_fisher_analysis
    from sidestep_engine.models.gpu_utils import clear_device_cache
    print(f"[INFO] Running PP++ on: {_ds}")
    print(f"       Rank budget: {pp_base_rank} (range {pp_rank_min}--{pp_rank_max})")
    try:
        r = run_fisher_analysis(checkpoint_dir=CHECKPOINT_DIR, variant=model_variant, dataset_dir=_ds,
            base_rank=pp_base_rank, rank_min=pp_rank_min, rank_max=pp_rank_max,
            timestep_focus=pp_timestep_focus, auto_confirm=True)
        if r:
            n = len(r.get("rank_pattern",{})); b = r.get("rank_budget",{})
            print(f"\n[OK] PP++: {n} modules, ranks {b.get('min','?')}--{b.get('max','?')}")
        else: print("[WARN] Analysis produced no results.")
    except Exception as e: print(f"[FAIL] {e}"); import traceback; traceback.print_exc()
    finally: clear_device_cache("cuda:0")
else:
    print("[INFO] Skipped.")


---
## S5 — Training
Train your adapter.

> Start with defaults, then tweak. Use the stop button to interrupt gracefully.
>
> **Resume a run:** Set `resume_from` to a checkpoint path on Drive.


In [ ]:
#@title S5a -- Training Configuration { display-mode: "form" }

#@markdown ---
#@markdown ### Required
dataset_dir = "/content/drive/MyDrive/SideStep/tensors/my_dataset"  #@param {type:"string"} {tooltip: "Path to .pt tensors (S3 output) or raw audio folder."}
output_dir  = "/content/drive/MyDrive/SideStep/adapters/my_lora"    #@param {type:"string"} {tooltip: "Where to save adapter weights."}
run_name    = ""  #@param {type:"string"} {tooltip: "Run name for TensorBoard. Blank = auto."}

#@markdown ---
#@markdown ### Adapter
adapter_type   = "lora"  #@param ["lora", "dora", "lokr", "loha", "oft"] {tooltip: "'lora'/'dora' = PEFT (PP++ compatible). 'lokr'/'loha' = LyCORIS."}
rank           = 64      #@param {type:"integer"} {tooltip: "LoRA rank. 32-128 typical."}
alpha          = 128     #@param {type:"integer"} {tooltip: "LoRA alpha. Convention: 2x rank."}
dropout        = 0.1     #@param {type:"number"} {tooltip: "Dropout. 0.05-0.15 prevents overfitting."}
attention_type = "both"  #@param ["both", "self", "cross"] {tooltip: "'both' = self + cross attention (recommended)."}
target_mlp     = True    #@param {type:"boolean"} {tooltip: "Also target MLP/FFN layers."}

#@markdown ---
#@markdown ### Training
learning_rate         = 1e-4  #@param {type:"number"} {tooltip: "LR. 1e-4 default. 5e-5 for PP++."}
batch_size            = 1     #@param {type:"integer"} {tooltip: "Samples per forward pass."}
gradient_accumulation = 4     #@param {type:"integer"} {tooltip: "Accumulate N steps before update."}
epochs                = 100   #@param {type:"integer"} {tooltip: "Full dataset passes."}
warmup_steps          = 100   #@param {type:"integer"} {tooltip: "LR ramp steps."}
max_steps             = 0     #@param {type:"integer"} {tooltip: "Hard stop after N steps. 0 = epochs only."}
seed                  = 42    #@param {type:"integer"} {tooltip: "Random seed."}

#@markdown ---
#@markdown ### Checkpointing
save_every      = 10    #@param {type:"integer"} {tooltip: "Checkpoint every N epochs."}
save_best       = True  #@param {type:"boolean"} {tooltip: "Keep best-loss checkpoint."}
log_every       = 10    #@param {type:"integer"} {tooltip: "TensorBoard log every N steps."}
log_heavy_every = 50    #@param {type:"integer"} {tooltip: "Heavy metrics every N steps. 0 = off."}
resume_from     = ""    #@param {type:"string"} {tooltip: "Checkpoint path to resume. Blank = fresh."}

#@markdown ---
#@markdown ### Advanced
optimizer_type         = "adamw"   #@param ["adamw", "adamw8bit", "prodigy", "sgd"] {tooltip: "'adamw' standard. 'adamw8bit' saves VRAM."}
scheduler_type         = "cosine"  #@param ["cosine", "linear", "constant", "cosine_with_restarts"] {tooltip: "'cosine' recommended."}
gradient_checkpointing = True      #@param {type:"boolean"} {tooltip: "Trade compute for VRAM. Essential on T4."}
offload_encoder        = False     #@param {type:"boolean"} {tooltip: "Offload text encoder to CPU. Saves ~2GB."}
weight_decay           = 0.01      #@param {type:"number"} {tooltip: "L2 regularization."}
max_grad_norm          = 1.0       #@param {type:"number"} {tooltip: "Gradient clipping."}
loss_weighting         = "none"    #@param ["none", "snr", "min_snr"] {tooltip: "Loss weighting strategy."}
cfg_ratio              = 0.15      #@param {type:"number"} {tooltip: "CFG dropout ratio."}
dataset_repeats        = 1         #@param {type:"integer"} {tooltip: "Repeat dataset N times per epoch."}

import os, datetime
from pathlib import Path
from IPython.display import HTML, display

if "_preprocessed_tensor_dir" in dir() and _preprocessed_tensor_dir:
    dataset_dir = _preprocessed_tensor_dir

if not run_name:
    _base = Path(dataset_dir).name if dataset_dir else "run"
    run_name = f"{_base}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"

if output_dir.endswith("my_lora"):
    output_dir = os.path.join(SIDESTEP_ROOT, "adapters", adapter_type, run_name)

_log_dir = os.path.join(SIDESTEP_ROOT, "logs", run_name)
os.makedirs(_log_dir, exist_ok=True)

_params = dict(
    checkpoint_dir=CHECKPOINT_DIR, model_variant=model_variant, base_model=model_variant,
    dataset_dir=dataset_dir, output_dir=output_dir, adapter_type=adapter_type,
    rank=rank, alpha=alpha, dropout=dropout, attention_type=attention_type, target_mlp=target_mlp,
    learning_rate=learning_rate, batch_size=batch_size, gradient_accumulation=gradient_accumulation,
    epochs=epochs, warmup_steps=warmup_steps, max_steps=max_steps, seed=seed,
    save_every=save_every, save_best=save_best, log_every=log_every, log_heavy_every=log_heavy_every,
    log_dir=_log_dir, run_name=run_name, resume_from=resume_from or None,
    optimizer_type=optimizer_type, scheduler_type=scheduler_type,
    gradient_checkpointing=gradient_checkpointing, offload_encoder=offload_encoder,
    weight_decay=weight_decay, max_grad_norm=max_grad_norm, loss_weighting=loss_weighting,
    cfg_ratio=cfg_ratio, dataset_repeats=dataset_repeats, device="cuda:0", precision="auto",
)

from sidestep_engine.cli.common import build_configs_from_dict
adapter_cfg, train_cfg = build_configs_from_dict(_params)

_SS = {"panel":"#2D2E36","border":"#4A5F8F","primary":"#5ccfe6","text":"#A1A8CF","muted":"#8995BC","bold":"#D5D8EC"}
display(HTML(
    f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
    f'border-left:4px solid {_SS["primary"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
    f'<div style="color:{_SS["primary"]};font-weight:bold;margin-bottom:8px;">Training Config</div>'
    f'<table style="border-collapse:collapse;">'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Adapter</td><td style="color:{_SS["bold"]};"><b>{adapter_type}</b> (rank={rank}, alpha={alpha})</td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Model</td><td style="color:{_SS["bold"]};"><b>{model_variant}</b></td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Dataset</td><td style="color:{_SS["text"]};"><code>{dataset_dir}</code></td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Output</td><td style="color:{_SS["text"]};"><code>{output_dir}</code></td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Run</td><td style="color:{_SS["bold"]};"><b>{run_name}</b></td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">LR</td><td style="color:{_SS["text"]};">{learning_rate} ({optimizer_type} + {scheduler_type})</td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Batch</td><td style="color:{_SS["text"]};">{batch_size} x {gradient_accumulation} accum</td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Epochs</td><td style="color:{_SS["text"]};">{epochs}{f" (max {max_steps} steps)" if max_steps else ""}</td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Logs</td><td style="color:{_SS["text"]};"><code>{_log_dir}</code></td></tr>'
    f'</table></div>'
))
print("[OK] Config ready. Run next cell to train.")


In [ ]:
#@title S5b -- Start Training { display-mode: "form" }
#@markdown Press the stop button to interrupt gracefully.

from tqdm.notebook import tqdm
from IPython.display import HTML, display
from sidestep_engine.models.loader import load_decoder_for_training
from sidestep_engine.core.trainer import FixedLoRATrainer
from sidestep_engine.models.gpu_utils import clear_device_cache

model = None; trainer = None
try:
    print(f"[INFO] Loading model (variant={train_cfg.model_variant}, device={train_cfg.device})...")
    model = load_decoder_for_training(
        checkpoint_dir=train_cfg.checkpoint_dir, variant=train_cfg.model_variant,
        device=train_cfg.device, precision=train_cfg.precision,
    )
    print("[OK] Model loaded.")

    trainer = FixedLoRATrainer(model, adapter_cfg, train_cfg)
    _pbar = tqdm(total=train_cfg.max_epochs, desc="Training", unit="epoch")
    _cur_epoch = 0; _last_loss = 0.0

    for update in trainer.train():
        step, loss, msg = update.step, update.loss, update.msg
        if hasattr(update, "epoch") and update.epoch > _cur_epoch:
            _cur_epoch = update.epoch; _pbar.update(1)
        _last_loss = loss
        _pbar.set_postfix({"step": step, "loss": f"{loss:.6f}", "lr": f"{update.lr:.2e}" if hasattr(update,"lr") else "?"})
        if hasattr(update, "kind"):
            if update.kind == "checkpoint": print(f"\n[OK] Checkpoint saved: {update.checkpoint_path}")
            elif update.kind == "warn": print(f"\n[WARN] {msg}")
            elif update.kind == "complete": print(f"\n[OK] {msg}")
    _pbar.close()

    _SS = {"panel":"#2D2E36","border":"#4A5F8F","success":"#7ec97a","text":"#A1A8CF","bold":"#D5D8EC"}
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["success"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["success"]};font-weight:bold;">[OK] Training Complete</div>'
        f'<div style="color:{_SS["text"]};">Output: <code>{output_dir}</code></div>'
        f'<div style="color:{_SS["text"]};margin-top:4px;">Final loss: <b style="color:{_SS["bold"]};">{_last_loss:.6f}</b></div></div>'
    ))

except KeyboardInterrupt:
    print("\n[INFO] Training interrupted. Partial checkpoints saved.")
except Exception as exc:
    print(f"\n[FAIL] Training failed: {exc}")
    import traceback; traceback.print_exc()
finally:
    del trainer; del model
    clear_device_cache("cuda:0")


In [ ]:
#@title S5c -- TensorBoard { display-mode: "form" }
#@markdown View training curves inline.

%load_ext tensorboard
%tensorboard --logdir {_log_dir}
